# Task 2: SQL Injection and Secure Coding Lab
This notebook demonstrates how a SQL Injection vulnerability is introduced in Python web applications and how to resolve it using parameterized SQL queries.

### What is SQL Injection?
SQL Injection (SQLi) is a security vulnerability where an attacker can manipulate SQL queries sent to a database. It occurs when user inputs are directly concatenated or formatted into a database query string instead of being treated as separate parameters.

## Step 1: Setting up a Temporary In-Memory Database
We will initialize a mock SQLite database in memory (`:memory:`) containing a user table. This allows us to perform safe tests without altering external files.

In [ ]:
import sqlite3
from werkzeug.security import generate_password_hash, check_password_hash

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create the users table
cursor.execute("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL,
    password TEXT NOT NULL,
    password_hash TEXT NOT NULL
)
""")

# Insert mock users
# admin / admin123
# alice / alice789
cursor.execute("""
INSERT INTO users (username, password, password_hash)
VALUES ('admin', 'admin123', ?)
""", (generate_password_hash("admin123"),))

cursor.execute("""
INSERT INTO users (username, password, password_hash)
VALUES ('alice', 'alice789', ?)
""", (generate_password_hash("alice789"),))

conn.commit()
print("[*] In-memory database setup completed.")

## Step 2: The Vulnerable Query Pattern
Below is the code typical of vulnerable applications. It formats the variables `input_user` and `input_pass` directly inside the string. 

In [ ]:
def vulnerable_login(input_user, input_pass):
    # VULNERABLE: String formatting creates vulnerability
    query = f"SELECT * FROM users WHERE username = '{input_user}' AND password = '{input_pass}'"
    print(f"Executing SQL Query: {query}")
    
    cursor.execute(query)
    result = cursor.fetchone()
    if result:
        print(f"[+] Login successful! Logged in as: {result[1]}\n")
    else:
        print("[-] Login failed: Invalid credentials.\n")

### Testing the Vulnerable Login
Let's test normal input vs an injection payload. The injection payload `admin' OR '1'='1` breaks out of the username field container, changing the execution flow of the database statement.

In [ ]:
print("--- Test 1: Normal Valid Credentials ---")
vulnerable_login("admin", "admin123")

print("--- Test 2: SQL Injection Attack Payload ---")
# Note: The username ends with a quote, injecting "OR '1'='1"
vulnerable_login("admin' OR '1'='1", "wrong_password")

## Step 3: The Secure (Remediated) Query Pattern
To fix this vulnerability, we rewrite the code to use **parameterized queries**. Instead of building the query dynamically, we use placeholders (`?`) and pass user inputs as parameters in a tuple.

In [ ]:
def secure_login(input_user, input_pass):
    # SECURE: Using placeholders
    query = "SELECT password_hash FROM users WHERE username = ?"
    print(f"Executing SQL Query: {query} (with parameters: {(input_user,)})")
    
    cursor.execute(query, (input_user,))
    row = cursor.fetchone()
    
    if row:
        stored_hash = row[0]
        # Securely compare hashes
        if check_password_hash(stored_hash, input_pass):
            print(f"[+] Login successful! Logged in as: {input_user}\n")
            return
            
    print("[-] Login failed: Invalid credentials.\n")

### Testing the Secure Login
Let's check if the injection bypass still works in the secure code.

In [ ]:
print("--- Test 3: Normal Valid Credentials (Secure) ---")
secure_login("admin", "admin123")

print("--- Test 4: SQL Injection Payload (Secure) ---")
# The query checks for a username exactly matching "admin' OR '1'='1", which fails.
secure_login("admin' OR '1'='1", "wrong_password")

## Conclusion & Key Takeaways
1. **Never concatenate strings** to build SQL. Use parameterized queries/prepared statements.
2. **Use Password Hashing** (e.g. PBKDF2, bcrypt, or Argon2) to store credentials instead of plaintext.
3. **Hide Database Error Details** from final users. Return generic errors like "Authentication Failed" to prevent attackers from mapping the database structure.